# Beacon API — Endpoint documentation

This notebook lists every HTTP endpoint the Beacon backend exposes. Those same URLs and payloads are what the mobile app uses: together they are a map of how the app talks to the server (sign-in, chat, evacuation route, checklist, onboarding, translations, etc.).

Use this as a step-by-step guide with Swagger: open each path, fill parameters, try **Execute** where possible.

Interactive docs (Swagger) : [Open `/docs`](https://beacon-api-396652353766.us-west1.run.app/docs)

**How to use this file**
1. Read each section for what the endpoint does in the app.
2. Use the linked Swagger page when you want to try a request live.


### 1. Authentication (`auth_api`)

#### `POST /auth/register`


- **Summary** : Create a user; returns `user_id`. 
- **Body** : `name`, `email`, `password`, `address`(optional)
- **Typical errors** : `409` if email already registered 

**Swagger screenshot:**  
![image](ss/register.png)

---

#### `POST /auth/login`


- **Summary** : Validate credentials; returns `user_id`.
- **Body** : `email`, `password` 
- **Typical errors** : `401` invalid email or password 

**Swagger screenshot:** 
![image](ss/login.png) 


### 2. Chat (`chatbot_api`)

#### `POST /chat/session/start`

- **Summary** : Start a chat session with GPS context (evac lookup, system prompt). Returns `session_id`, `location`, `evac_data`, `geojson`.
- **Body** : `lat`, `lon`, optional `timestamp`, `user_id`, route/search params (`distance`, etc.)

**Swagger screenshot:**  



#### `GET /chat/session/{session_id}`

- **Summary** : Load persisted session state (history, location, evac).
- **Path** : `session_id` 

**Swagger screenshot:**  



#### `POST /chat/message`

- **Summary** : Send a user message; **streams** SSE (`text/event-stream`) with assistant reply chunks and optional `language` event.
- **Body** : `session_id`, `message` 

**Swagger screenshot:**  


### 3. Maps & routing (`maps_api`)

#### `GET /route/generate`

- **Summary** : Find evacuation route to a shelter candidate; returns GeoJSON string or `no_routes`.
- **Query** : `lat`, `lon`, `timestamp`, optional `dropby_type`, `distance`, `hwp_threshold`, `hwp_max_fraction`, `max_candidates`, `language`

**Swagger screenshot:**

### 4. UI translation (`translate_api`)

#### `POST /translate`

- **Summary** : Batch-translate English UI strings to a target language (used by the mobile app).
- **Body** : `language` (ISO 639-1), `strings` (list of English strings)
- **Response** : JSON object mapping each original string to its translated string

**Swagger screenshot:**

### 5. Checklist (`checklist_api`)

#### `GET /checklist`

- **Summary** : Checklist tab payload: `mode` plus `categories` (onboarding vs evacuation).
- **Query** : Optional `user_id`

**Swagger screenshot:**

---

#### `PATCH /checklist/item`

- **Summary** : Persist checked state for an evacuation checklist item.
- **Body** : `item_id`, `checked`, optional `user_id`

**Swagger screenshot:**

---

#### `POST /generate`

- **Summary** : **Preview** personalized checklist from onboarding answers (no DB required for the core generation).
- **Body** : `OnboardingAnswers` (booleans: `owns_home`, `has_car`, …)

**Swagger screenshot:**

---

#### `GET /me`

- **Summary** : Legacy shape: `summary` plus `checklist` dict for a user. Prefer `GET /checklist` for the tab UI.
- **Query** : `user_id`

**Swagger screenshot:**

### 6. Onboarding (`onboarding_api`)

All paths are under `/onboarding` (tag **onboarding** in Swagger).

#### `GET /onboarding/status`

- **Summary** : Whether the user has completed household onboarding (`completed` boolean).
- **Query** : `user_id`

**Swagger screenshot:**

---

#### `POST /onboarding/household`

- **Summary** : Create or update household answers (upsert).
- **Query** : `user_id`
- **Body** : `HouseholdAnswers` (booleans)
- **Success** : `201 Created`

**Swagger screenshot:**

---

#### `GET /onboarding/household`

- **Summary** : Load saved household answers.
- **Query** : `user_id`
- **Typical errors** : `404` if no household row

**Swagger screenshot:**

### 7. Monitor & push (`monitor_api`)

#### `POST /monitor/register_fcm_token`

- **Summary** : Register or update FCM device token for push notifications.
- **Query** : `user_id`, `device_token`, optional `platform` (default `ios`)

**Swagger screenshot:**

---

#### `GET /monitor/evac`

- **Summary** : Evacuation zone records for a location; may trigger push if `user_id` and token exist.
- **Query** : `lat`, `lon`, `timestamp`, optional `user_id`

**Swagger screenshot:**

---

#### `GET /monitor/hwp`

- **Summary** : Hourly wildfire potential (HWP) records; may send fire-danger push if over threshold.
- **Query** : `lat`, `lon`, `timestamp`, optional `hwp_threshold`, `user_id`

**Swagger screenshot:**

---

#### Not exposed in code (for reference)

- `GET /monitor/fire` exists in source as **commented out** — not active unless uncommented and redeployed.

### 8. Quick reference — all paths

- `POST /auth/register`
- `POST /auth/login`
- `POST /chat/session/start`
- `GET /chat/session/{session_id}`
- `POST /chat/message`
- `GET /route/generate`
- `POST /translate`
- `GET /checklist`
- `PATCH /checklist/item`
- `POST /generate`
- `GET /me`
- `GET /onboarding/status`
- `POST /onboarding/household`
- `GET /onboarding/household`
- `POST /monitor/register_fcm_token`
- `GET /monitor/evac`
- `GET /monitor/hwp`

Also: **GET** `/docs`, **GET** `/openapi.json`, **GET** `/redoc` (FastAPI defaults).